# v3 — Phase 1 : Video Inference

逐格跑完整支影片，將偵測到的 frame 依照 REVIEW_EVERY 間距存到 `{patient_ID}/tmp/`。

- `frame_id < KEYFRAME`  → 每 `REVIEW_EVERY_before` 張偵測存 1 張
- `frame_id >= KEYFRAME` → 每 `REVIEW_EVERY_after`  張偵測存 1 張

跑完後請執行 **v3_phace_2.ipynb** 來 review。

In [1]:
from pathlib import Path
import os

# ── Model ─────────────────────────────────────────────────────────────
MODEL_PATH = Path("/datadrive/polyp/yolo_output/runs/detect/2026_04_15(1)/weights/best.pt")

# ── Input videos ──────────────────────────────────────────────────────
VIDEO_DIR  = Path("videos/")

# ── Output root ───────────────────────────────────────────────────────
OUT_ROOT   = Path("pre_v3")

# ── Inference settings ────────────────────────────────────────────────
CONF_THRESH = 0.5
BATCH_SIZE  = 512

# ── Parallelism ───────────────────────────────────────────────────────
N_PARALLEL  = 1   # ← 同時處理幾支影片（frame 讀取 + 存檔並行，GPU inference 序列化）

# ── Review settings ───────────────────────────────────────────────────
KEYFRAME            = 3000
REVIEW_EVERY_before = 500
REVIEW_EVERY_after  = 100

print(f"Model      : {MODEL_PATH}")
print(f"Video dir  : {VIDEO_DIR}")
print(f"Output root: {OUT_ROOT.resolve()}")
print(f"CPU cores  : {os.cpu_count()}")
print(f"Parallel   : {N_PARALLEL} videos at once")
print(f"Keyframe   : {KEYFRAME}")

Model      : /datadrive/polyp/yolo_output/runs/detect/2026_04_15(1)/weights/best.pt
Video dir  : videos
Output root: /root/Desktop/polyp/yolo_data/pre_v3
CPU cores  : 192
Parallel   : 1 videos at once
Keyframe   : 3000


In [2]:
import cv2
import torch
import numpy as np
import threading
import queue
from concurrent.futures import ThreadPoolExecutor, as_completed
from ultralytics import YOLO
from tqdm import tqdm

# DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
DEVICE = "cuda"
print(f"Device: {DEVICE}")

# Serialize GPU calls when multiple videos run in parallel
_gpu_lock = threading.Lock()

def get_scope_bbox(frame: np.ndarray):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    k25 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))
    k15 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k25)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k15)
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    return cv2.boundingRect(max(cnts, key=cv2.contourArea))

def crop_scope(frame: np.ndarray, bbox) -> np.ndarray:
    x, y, w, h = bbox
    return frame[y:y+h, x:x+w]

def patient_id_from_path(p: Path) -> str:
    return p.stem.split("_")[0]

print("Helpers loaded.")

Device: cuda
Helpers loaded.


In [3]:
model = YOLO(str(MODEL_PATH))
model.to(DEVICE)
print(f"Model loaded : {MODEL_PATH.name}")
print(f"Classes      : {model.names}")

Model loaded : best.pt
Classes      : {0: 'non-adenoma', 1: 'adenoma'}


In [4]:
def infer_video(video_path: Path) -> dict:
    """
    Parallel-safe inference pipeline per video:
      - Reader thread   : video decode → frame_q  (CPU, fully parallel across videos)
      - Inference thread: YOLO predict with _gpu_lock (GPU serialized across videos)
      - Saver pool      : cv2.imwrite × 2 + txt   (I/O, fully parallel across videos)
    """
    pid = patient_id_from_path(video_path)
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps          = cap.get(cv2.CAP_PROP_FPS)

    print(f"\n{'='*60}")
    print(f"Video    : {video_path.name}")
    print(f"Patient  : {pid}")
    print(f"Frames   : {total_frames}  @ {fps:.1f} fps")

    scope_bbox = None
    for _ in range(min(60, total_frames)):
        ok, fr = cap.read()
        if ok:
            scope_bbox = get_scope_bbox(fr)
            if scope_bbox:
                break
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

    if scope_bbox:
        bx, by, bw, bh = scope_bbox
        print(f"Scope    : {bw}×{bh} (x={bx}, y={by})", flush=True)
    else:
        print("WARNING: scope bbox not found — using full frame", flush=True)

    tmp_dir = OUT_ROOT / pid / "tmp"
    tmp_dir.mkdir(parents=True, exist_ok=True)

    # ── helpers ──────────────────────────────────────────────────────
    def boxes_to_yolo(r) -> str:
        confs  = r.boxes.conf.cpu().numpy()
        best   = int(confs.argmax())
        cls    = int(r.boxes.cls[best].cpu().numpy())
        cx, cy, w, h = r.boxes.xywhn[best].cpu().numpy()
        return f"{cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"

    def flush_buf(buf):
        imgs = [item[1] for item in buf]
        with _gpu_lock:   # serialize GPU across parallel video threads
            res = model.predict(
                source=imgs, conf=CONF_THRESH, imgsz=640,
                device=DEVICE, verbose=False
            )
        hits = []
        for (fid, orig), r in zip(buf, res):
            if r.boxes is not None and len(r.boxes) > 0:
                hits.append((fid, orig, r.plot(), boxes_to_yolo(r)))
        return hits

    def save_to_tmp(hit):
        fid, orig, annotated, label_txt = hit
        stem = f"{pid}_{fid:06d}"
        cv2.imwrite(str(tmp_dir / f"{stem}.jpg"),     orig)
        cv2.imwrite(str(tmp_dir / f"{stem}_vis.jpg"), annotated)
        (tmp_dir / f"{stem}.txt").write_text(label_txt)

    # ── Reader thread ────────────────────────────────────────────────
    frame_q  = queue.Queue(maxsize=BATCH_SIZE * 8)
    read_err = [None]

    def _reader():
        try:
            fid = 0
            while True:
                ok, raw = cap.read()
                if not ok:
                    break
                cropped = crop_scope(raw, scope_bbox) if scope_bbox else raw
                frame_q.put((fid, cropped))
                fid += 1
        except Exception as e:
            read_err[0] = e
        finally:
            frame_q.put(None)

    reader = threading.Thread(target=_reader, daemon=True)
    reader.start()

    # ── Saver pool ───────────────────────────────────────────────────
    save_pool    = ThreadPoolExecutor(max_workers=4)
    save_futures = []

    def submit_save(hit):
        save_futures.append(save_pool.submit(save_to_tmp, hit))
        return f"{pid}_{hit[0]:06d}"

    def drain_to_tmp(buf, every):
        saved = []
        while len(buf) >= every:
            saved.append(submit_save(buf[every - 1]))
            buf = buf[every:]
        return buf, saved

    # ── Inference loop ───────────────────────────────────────────────
    infer_buf      = []
    pre_buf        = []
    post_buf       = []
    frame_id       = 0
    total_detected = 0
    crossed        = False
    stems          = []
    LOG_EVERY      = max(1000, BATCH_SIZE * 2)

    while True:
        item = frame_q.get()
        if item is None:
            break
        infer_buf.append(item)
        frame_id += 1

        if len(infer_buf) >= BATCH_SIZE:
            hits = flush_buf(infer_buf)
            infer_buf = []
            total_detected += len(hits)

            for hit in hits:
                fid = hit[0]
                if fid < KEYFRAME:
                    pre_buf.append(hit)
                    pre_buf, saved = drain_to_tmp(pre_buf, REVIEW_EVERY_before)
                    stems.extend(saved)
                else:
                    if not crossed:
                        crossed = True
                        if pre_buf:
                            stems.append(submit_save(pre_buf[-1]))
                            pre_buf = []
                    post_buf.append(hit)
                    post_buf, saved = drain_to_tmp(post_buf, REVIEW_EVERY_after)
                    stems.extend(saved)

            if frame_id % LOG_EVERY < BATCH_SIZE:
                pct = frame_id / total_frames * 100
                print(f"  [{pid}] {frame_id:>6}/{total_frames} ({pct:4.1f}%)  detected={total_detected}", flush=True)

    if infer_buf:
        hits = flush_buf(infer_buf)
        total_detected += len(hits)
        for hit in hits:
            fid = hit[0]
            if fid < KEYFRAME:
                pre_buf.append(hit)
            else:
                if not crossed:
                    crossed = True
                    if pre_buf:
                        stems.append(submit_save(pre_buf[-1]))
                        pre_buf = []
                post_buf.append(hit)

    for buf in (pre_buf, post_buf):
        if buf:
            stems.append(submit_save(buf[-1]))

    for f in save_futures:
        f.result()
    save_pool.shutdown(wait=False)
    reader.join()
    cap.release()

    if read_err[0]:
        raise RuntimeError(f"[{pid}] Reader error: {read_err[0]}")

    print(f"  [{pid}] Done. {frame_id}/{total_frames} frames  detected={total_detected}  saved={len(stems)}", flush=True)
    return {"pid": pid, "frames": frame_id, "detected": total_detected, "saved_tmp": len(stems)}

print("infer_video() defined.")

infer_video() defined.


In [5]:
video_exts = ("*.MOV", "*.mov", "*.mp4", "*.MP4", "*.avi", "*.AVI")
video_files = []
for ext in video_exts:
    video_files.extend(VIDEO_DIR.glob(ext))
video_files = sorted(set(video_files))

print(f"Found {len(video_files)} video(s):")
for v in video_files:
    print(f"  {v.name}")

Found 13 video(s):
  R0118_20221017_100817.MOV
  R0119_20221017_105137.MOV
  R0120_20221017_105410.MOV
  R0122_20221017_113551.MOV
  R0123_20221017_114112.MOV
  R0124_20221017_122805.MOV
  R0125_20221017_124451.MOV
  R0126_20221017_130100.MOV
  R0127_20221020_100409.MOV
  R0128_20221020_102642.MOV
  R0129_20221020_104946.MOV
  R0130_20221020_112048.mp4
  R0131_20221020_114037.mp4


In [ ]:
todo = [vp for vp in video_files
        if not (OUT_ROOT / patient_id_from_path(vp) / "tmp").exists()]
skip = [patient_id_from_path(vp) for vp in video_files if vp not in todo]

if skip:
    print(f"Skipping (already done): {', '.join(skip)}")
print(f"To process: {len(todo)} video(s)  |  N_PARALLEL={N_PARALLEL}\n")

summary = []

with ThreadPoolExecutor(max_workers=N_PARALLEL) as pool:
    futs = {pool.submit(infer_video, vp): vp for vp in todo}
    for fut in tqdm(as_completed(futs), total=len(futs), desc="Videos done"):
        try:
            summary.append(fut.result())
        except Exception as e:
            print(f"\nERROR in {futs[fut].name}: {e}")

print("\n" + "="*60)
print("SUMMARY")
print(f"{'Patient':<12} {'Frames':>8} {'Detected':>10} {'Saved→tmp':>10}")
print("-" * 44)
for s in sorted(summary, key=lambda x: x["pid"]):
    print(f"{s['pid']:<12} {s['frames']:>8} {s['detected']:>10} {s['saved_tmp']:>10}")
print("-" * 44)
print(f"{'TOTAL':<12} {sum(s['frames'] for s in summary):>8}"
      f"{sum(s['detected'] for s in summary):>10} "
      f"{sum(s['saved_tmp'] for s in summary):>10}")
print("\nDone. Now open v3_phase_2.ipynb to label the frames.")

To process: 13 video(s)  |  N_PARALLEL=1



Videos done:   0%|          | 0/13 [00:00<?, ?it/s]


Video    : R0118_20221017_100817.MOV
Patient  : R0118
Frames   : 40815  @ 60.0 fps
Scope    : 1163×1009 (x=717, y=36)
  [R0118]   1024/40815 ( 2.5%)  detected=121
  [R0118]   2048/40815 ( 5.0%)  detected=276
  [R0118]   3072/40815 ( 7.5%)  detected=397
  [R0118]   4096/40815 (10.0%)  detected=470
  [R0118]   5120/40815 (12.5%)  detected=542
